# Mini-Project: Soccer Results Analytics
## Part 1 - Historical Data Preparation
**Objectif:** Charger, nettoyer et enrichir les données historiques
du championnat mauritanien de football (2007-2025).

**Dataset:** rim_championnat_results_2007-2025.csv

**Auteur:** Raghye Meissara Bilal

## SparkSession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("SoccerDataPreparation") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f" Spark {spark.version} prêt!")

 Spark 3.5.3 prêt!


## Part 1 — Historical Data Preparation

## Charger les données

In [3]:
df = spark.read.csv(
    "/workspace/data/rim_championnat.csv",
    header=True,
    inferSchema=True
)

print("=== Schema brut ===")
df.printSchema()
print(f"\nTotal lignes: {df.count()}")
df.show(10, truncate=False)

=== Schema brut ===
root
 |-- season: string (nullable = true)
 |-- date: string (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: integer (nullable = true)
 |-- away_goals: integer (nullable = true)


Total lignes: 2744
+------------------------+----------+---------------+---------------+----------+----------+
|season                  |date      |home_team      |away_team      |home_goals|away_goals|
+------------------------+----------+---------------+---------------+----------+----------+
|Championnat D1 2007/2008|02.08.2008|ASAC Concorde  |Nasr Sebkha    |2         |0         |
|Championnat D1 2007/2008|02.08.2008|Police         |Tidjikja       |0         |1         |
|Championnat D1 2007/2008|01.08.2008|Ksar           |SNIM           |1         |0         |
|Championnat D1 2007/2008|01.08.2008|Mauritel Mobile|Armee          |0         |0         |
|Championnat D1 2007/2008|01.08.2008|Tevragh-Zeina  |FC Trarza      

## Nettoyage et colonnes dérivées

In [4]:
# Nettoyer et enrichir les données
df_clean = df \
    .na.drop(subset=["home_team", "away_team", "home_goals", "away_goals"]) \
    .withColumn("total_goals", col("home_goals") + col("away_goals")) \
    .withColumn("goal_difference", col("home_goals") - col("away_goals")) \
    .withColumn("result", 
        when(col("home_goals") > col("away_goals"), "home_win")
        .when(col("home_goals") == col("away_goals"), "draw")
        .otherwise("away_win")
    ) \
    .withColumn("match_year", 
        regexp_extract(col("season"), r"(\d{4})", 1).cast("int")
    ) \
    .withColumn("is_high_scoring",
        when(col("total_goals") >= 4, 1).otherwise(0)
    )

print(f" Lignes après nettoyage: {df_clean.count()}")
print("\n=== Distribution des résultats ===")
df_clean.groupBy("result").count().orderBy("count", ascending=False).show()

print("\n=== Statistiques des buts ===")
df_clean.select(
    avg("total_goals").alias("avg_goals"),
    max("total_goals").alias("max_goals"),
    min("total_goals").alias("min_goals")
).show()

df_clean.show(5, truncate=False)

 Lignes après nettoyage: 2744

=== Distribution des résultats ===
+--------+-----+
|  result|count|
+--------+-----+
|home_win| 1086|
|away_win|  901|
|    draw|  757|
+--------+-----+


=== Statistiques des buts ===
+-----------------+---------+---------+
|        avg_goals|max_goals|min_goals|
+-----------------+---------+---------+
|2.314139941690962|       11|        0|
+-----------------+---------+---------+

+------------------------+----------+---------------+-----------+----------+----------+-----------+---------------+--------+----------+---------------+
|season                  |date      |home_team      |away_team  |home_goals|away_goals|total_goals|goal_difference|result  |match_year|is_high_scoring|
+------------------------+----------+---------------+-----------+----------+----------+-----------+---------------+--------+----------+---------------+
|Championnat D1 2007/2008|02.08.2008|ASAC Concorde  |Nasr Sebkha|2         |0         |2          |2              |home_win|20

## Sauvegarder en Parquet

In [5]:
# Sauvegarder en Parquet partitionné par match_year
output_path = "/workspace/data/outputs/prepared"

df_clean.write \
    .mode("overwrite") \
    .partitionBy("match_year") \
    .parquet(output_path)

print(f" Données sauvegardées en Parquet!")
print(f"   Path: {output_path}")

# Vérifier
import subprocess
result = subprocess.run(
    ["find", output_path, "-name", "*.parquet", "-type", "f"],
    capture_output=True, text=True
)
files = result.stdout.strip().split("\n")
print(f"   Fichiers créés: {len([f for f in files if f])}")

# Relire pour vérifier
df_verify = spark.read.parquet(output_path)
print(f"   Lignes vérifiées: {df_verify.count()}")
df_verify.show(3, truncate=False)

 Données sauvegardées en Parquet!
   Path: /workspace/data/outputs/prepared
   Fichiers créés: 17
   Lignes vérifiées: 2744
+-----------------+------+---------------+-----------------+----------+----------+-----------+---------------+--------+---------------+----------+
|season           |date  |home_team      |away_team        |home_goals|away_goals|total_goals|goal_difference|result  |is_high_scoring|match_year|
+-----------------+------+---------------+-----------------+----------+----------+-----------+---------------+--------+---------------+----------+
|Ligue 1 2024/2025|30.05.|Garde Nationale|Nouakchott King's|1         |1         |2          |0              |draw    |0              |2024      |
|Ligue 1 2024/2025|30.05.|Toulde         |Ksar             |3         |0         |3          |3              |home_win|0              |2024      |
|Ligue 1 2024/2025|29.05.|Kaedi          |Douane           |2         |0         |2          |2              |home_win|0              |2024  

## Part 2 — Current Season Enrichment by Web Scraping

### Web Scraping

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Essayer différentes URLs Wikipedia
urls = [
    "https://en.wikipedia.org/wiki/2024-25_Mauritanian_Ligue_1",
    "https://en.wikipedia.org/wiki/Mauritanian_Ligue_1",
    "https://fr.wikipedia.org/wiki/Ligue_1_mauritanienne_de_football_2024-2025",
    "https://fr.wikipedia.org/wiki/Championnat_de_Mauritanie_de_football_2024-2025",
]

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

for url in urls:
    try:
        response = requests.get(url, headers=headers, timeout=15)
        print(f"Status {response.status_code}: {url}")
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            tables = soup.find_all('table', class_='wikitable')
            print(f"  → Tables wikitable: {len(tables)}")
    except Exception as e:
        print(f"Erreur: {e}")

Status 404: https://en.wikipedia.org/wiki/2024-25_Mauritanian_Ligue_1
Status 404: https://en.wikipedia.org/wiki/Mauritanian_Ligue_1
Status 404: https://fr.wikipedia.org/wiki/Ligue_1_mauritanienne_de_football_2024-2025
Status 200: https://fr.wikipedia.org/wiki/Championnat_de_Mauritanie_de_football_2024-2025
  → Tables wikitable: 2


In [7]:
%pip install lxml html5lib --user

Note: you may need to restart the kernel to use updated packages.


In [8]:
import lxml
print("L'import de lxml a réussi !")

L'import de lxml a réussi !


In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://fr.wikipedia.org/wiki/Championnat_de_Mauritanie_de_football_2024-2025"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(url, headers=headers, timeout=15)
soup = BeautifulSoup(response.content, 'html.parser')

tables = soup.find_all('table', class_='wikitable')
print(f"Tables trouvées: {len(tables)}")

for i, table in enumerate(tables):
    print(f"\n=== Table {i+1} ===")
    try:
        df_wiki = pd.read_html(str(table), flavor='html5lib')[0]
        print(df_wiki.head(10))
        print(f"Colonnes: {list(df_wiki.columns)}")
        print(f"Shape: {df_wiki.shape}")
    except Exception as e:
        print(f"Erreur: {e}")

Tables trouvées: 2

=== Table 1 ===
   Rang               Équipe  Pts   J   G   N   P  Bp  Bc  Diff
0     1             Al Hilal   67  30  20   7   3  55  18    37
1     2      FC Nouadhibou T   61  30  17  10   3  37  13    24
2     3            Chemal FC   51  30  14   9   7  37  23    14
3     4    Nouakchott King's   50  30  12  14   4  36  25    11
4     5           AS Douanes   49  30  12  13   5  39  30     9
5     6       Al Merreikh SC   44  30  12   8  10  36  28     8
6     7     FC Tevragh Zeïna   43  30  11  10   9  35  26     9
7     8          AS Pompiers   38  30   9  11  10  31  30     1
8     9  FC Inter Nouakchott   37  30   9  10  11  31  37    -6
9    10             Kaédi FC   36  30   9   9  12  37  49   -12
Colonnes: ['Rang', 'Équipe', 'Pts', 'J', 'G', 'N', 'P', 'Bp', 'Bc', 'Diff']
Shape: (16, 10)

=== Table 2 ===
                                                 0  \
0  Championnat de Mauritanie de football 2024-2025   
1                                         C

## Traitement données scrapées

In [10]:
from pyspark.sql.types import *

# Table 1 = Classement Ligue 1 2024/2025
df_standing = pd.read_html(str(tables[0]), flavor='html5lib')[0]

print("=== Classement Wikipedia scraped ===")
print(df_standing)

# Convertir en Spark DataFrame
df_scraped_spark = spark.createDataFrame(df_standing) \
    .withColumnRenamed("Rang", "rank") \
    .withColumnRenamed("Équipe", "team") \
    .withColumnRenamed("Pts", "points") \
    .withColumnRenamed("J", "played") \
    .withColumnRenamed("G", "wins") \
    .withColumnRenamed("N", "draws") \
    .withColumnRenamed("P", "losses") \
    .withColumnRenamed("Bp", "goals_for") \
    .withColumnRenamed("Bc", "goals_against") \
    .withColumnRenamed("Diff", "goal_diff") \
    .withColumn("season", lit("Ligue 1 2024/2025")) \
    .withColumn("source", lit("Wikipedia FR - scraped"))

print("\n=== Données scrapées en Spark ===")
df_scraped_spark.show(truncate=False)

# Sauvegarder
scraped_path = "/workspace/data/outputs/scraped"
df_scraped_spark.write \
    .mode("overwrite") \
    .parquet(scraped_path)

print(f"\n Classement scraped sauvegardé!")
print(f"   Source: https://fr.wikipedia.org/wiki/Championnat_de_Mauritanie_de_football_2024-2025")
print(f"   Équipes: {df_scraped_spark.count()}")

=== Classement Wikipedia scraped ===
    Rang               Équipe  Pts   J   G   N   P  Bp  Bc  Diff
0      1             Al Hilal   67  30  20   7   3  55  18    37
1      2      FC Nouadhibou T   61  30  17  10   3  37  13    24
2      3            Chemal FC   51  30  14   9   7  37  23    14
3      4    Nouakchott King's   50  30  12  14   4  36  25    11
4      5           AS Douanes   49  30  12  13   5  39  30     9
5      6       Al Merreikh SC   44  30  12   8  10  36  28     8
6      7     FC Tevragh Zeïna   43  30  11  10   9  35  26     9
7      8          AS Pompiers   38  30   9  11  10  31  30     1
8      9  FC Inter Nouakchott   37  30   9  10  11  31  37    -6
9     10             Kaédi FC   36  30   9   9  12  37  49   -12
10    11          ASC Gendrim   34  30   9   7  14  24  35   -11
11    12             ASC SNIM   32  30   8   8  14  24  28    -4
12    13         FC Nzidane P   30  30   6  12  12  31  42   -11
13    14              FC Ksar   29  30   7   8  15  1

## Sauvegarder prepare_data et résumé

In [11]:
# Sauvegarder les données préparées en Parquet
output_path = "/workspace/data/outputs/prepared"

df_clean.write \
    .mode("overwrite") \
    .partitionBy("match_year") \
    .parquet(output_path)

print(" Résumé Part 1 & 2:")
print(f"   Données historiques : {df_clean.count()} matchs")
print(f"   Saisons             : {df_clean.select('season').distinct().count()}")
print(f"   Années              : 2007-2025")
print(f"   Format              : Parquet partitionné par match_year")
print(f"\n Données scrapées (Part 2):")
print(f"   Source  : Wikipedia FR")
print(f"   URL     : https://fr.wikipedia.org/wiki/Championnat_de_Mauritanie_de_football_2024-2025")
print(f"   Équipes : 16")
print(f"   Format  : Parquet")
print(f"\n prepare_data.ipynb terminé!")

 Résumé Part 1 & 2:
   Données historiques : 2744 matchs
   Saisons             : 18
   Années              : 2007-2025
   Format              : Parquet partitionné par match_year

 Données scrapées (Part 2):
   Source  : Wikipedia FR
   URL     : https://fr.wikipedia.org/wiki/Championnat_de_Mauritanie_de_football_2024-2025
   Équipes : 16
   Format  : Parquet

 prepare_data.ipynb terminé!


 ## Part 3 — Real-Time Stream Simulation

## Créer topic + Lancer producteur

In [12]:
import subprocess, json, time
%pip install kafka-python-ng
print(" kafka-python-ng installé!")
from kafka import KafkaProducer

# Créer le topic
subprocess.run([
    "bash", "-c",
    "docker exec kafka /opt/kafka/bin/kafka-topics.sh --create \
    --bootstrap-server localhost:9092 \
    --topic soccer_matches \
    --partitions 3 \
    --replication-factor 1 2>/dev/null || echo 'exists'"
], capture_output=True)
print(" Topic soccer_matches prêt!")

# Lire les données
df_prod = spark.read.parquet("/workspace/data/outputs/prepared")
matches = df_prod.select(
    "season", "date", "home_team", "away_team",
    "home_goals", "away_goals", "result",
    "total_goals", "match_year"
).collect()
print(f" {len(matches)} matchs à envoyer!")

# Producteur Kafka
producer = KafkaProducer(
    bootstrap_servers='kafka:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

for i, row in enumerate(matches):
    message = {
        "season":      row["season"],
        "date":        str(row["date"]),
        "home_team":   row["home_team"],
        "away_team":   row["away_team"],
        "home_goals":  row["home_goals"],
        "away_goals":  row["away_goals"],
        "result":      row["result"],
        "total_goals": row["total_goals"],
        "match_year":  row["match_year"]
    }
    producer.send('soccer_matches', value=message)
    if (i+1) % 200 == 0:
        print(f" {i+1}/{len(matches)} envoyés...")
    time.sleep(0.1)

producer.flush()
print(f" Tous les {len(matches)} matchs envoyés à Kafka!")

Note: you may need to restart the kernel to use updated packages.
 kafka-python-ng installé!
 Topic soccer_matches prêt!
 2744 matchs à envoyer!
 200/2744 envoyés...
 400/2744 envoyés...
 600/2744 envoyés...
 800/2744 envoyés...
 1000/2744 envoyés...
 1200/2744 envoyés...
 1400/2744 envoyés...
 1600/2744 envoyés...
 1800/2744 envoyés...
 2000/2744 envoyés...
 2200/2744 envoyés...
 2400/2744 envoyés...
 2600/2744 envoyés...
 Tous les 2744 matchs envoyés à Kafka!


##  Part 4 — Streaming Processing with Spark

In [13]:
from pyspark.sql.types import *
import time, subprocess

# Nettoyer les anciens checkpoints
subprocess.run(["rm", "-rf", "/workspace/data/checkpoints/streaming"], capture_output=True)
subprocess.run(["rm", "-rf", "/workspace/data/outputs/streaming"], capture_output=True)

schema = StructType([
    StructField("season",      StringType(),  True),
    StructField("date",        StringType(),  True),
    StructField("home_team",   StringType(),  True),
    StructField("away_team",   StringType(),  True),
    StructField("home_goals",  IntegerType(), True),
    StructField("away_goals",  IntegerType(), True),
    StructField("result",      StringType(),  True),
    StructField("total_goals", IntegerType(), True),
    StructField("match_year",  IntegerType(), True)
])

# Lire depuis Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "soccer_matches") \
    .option("startingOffsets", "earliest") \
    .load()

matches_df = kafka_df \
    .select(
        from_json(col("value").cast("string"), schema).alias("data"),
        col("timestamp")
    ) \
    .select("data.*", "timestamp") \
    .withWatermark("timestamp", "2 minutes")

# Points domicile
home_df = matches_df.select(
    col("home_team").alias("team"),
    when(col("home_goals") > col("away_goals"), 3)
        .when(col("home_goals") == col("away_goals"), 1)
        .otherwise(0).alias("points"),
    when(col("home_goals") > col("away_goals"), 1).otherwise(0).alias("wins"),
    when(col("home_goals") == col("away_goals"), 1).otherwise(0).alias("draws"),
    when(col("home_goals") < col("away_goals"), 1).otherwise(0).alias("losses"),
    col("home_goals").alias("gf"),
    col("away_goals").alias("ga"),
    lit(1).alias("played")
)

# Points extérieur
away_df = matches_df.select(
    col("away_team").alias("team"),
    when(col("away_goals") > col("home_goals"), 3)
        .when(col("away_goals") == col("home_goals"), 1)
        .otherwise(0).alias("points"),
    when(col("away_goals") > col("home_goals"), 1).otherwise(0).alias("wins"),
    when(col("away_goals") == col("home_goals"), 1).otherwise(0).alias("draws"),
    when(col("away_goals") < col("home_goals"), 1).otherwise(0).alias("losses"),
    col("away_goals").alias("gf"),
    col("home_goals").alias("ga"),
    lit(1).alias("played")
)

# Classement
all_df = home_df.union(away_df)
ranking_df = all_df.groupBy("team").agg(
    sum("played").alias("MJ"),
    sum("wins").alias("V"),
    sum("draws").alias("N"),
    sum("losses").alias("D"),
    sum("gf").alias("BP"),
    sum("ga").alias("BC"),
    (sum("gf") - sum("ga")).alias("GD"),
    sum("points").alias("PTS")
).orderBy(col("PTS").desc(), col("GD").desc())

# Ecrire en Parquet avec checkpoint
def write_ranking(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.write \
            .mode("overwrite") \
            .parquet("/workspace/data/outputs/streaming")
        print(f" Batch {batch_id} écrit - {batch_df.count()} équipes!")

query = ranking_df.writeStream \
    .foreachBatch(write_ranking) \
    .outputMode("complete") \
    .option("checkpointLocation", "/workspace/data/checkpoints/streaming") \
    .trigger(processingTime="30 seconds") \
    .start()

print(" Streaming démarré!")
time.sleep(90)
query.stop()
print(" Streaming arrêté!")

 Streaming démarré!
 Batch 0 écrit - 43 équipes!
 Streaming arrêté!


In [14]:
# Lire et afficher le classement streamé
ranking_result = spark.read.parquet("/workspace/data/outputs/streaming")

print("===  CLASSEMENT GÉNÉRAL - Championnat Mauritanien ===")
print(f"Total équipes: {ranking_result.count()}")
ranking_result.orderBy(col("PTS").desc(), col("GD").desc()).show(20, truncate=False)

===  CLASSEMENT GÉNÉRAL - Championnat Mauritanien ===
Total équipes: 43
+-----------------+---+---+---+---+---+---+----+---+
|team             |MJ |V  |N  |D  |BP |BC |GD  |PTS|
+-----------------+---+---+---+---+---+---+----+---+
|Nouadhibou       |414|252|98 |64 |689|260|429 |854|
|Tevragh-Zeina    |414|227|103|84 |660|305|355 |784|
|ASAC Concorde    |384|194|101|89 |608|363|245 |683|
|SNIM             |406|154|121|131|466|386|80  |583|
|Ksar             |414|142|123|149|470|478|-8  |549|
|Tidjikja         |322|131|93 |98 |407|320|87  |486|
|Nouakchott King's|296|100|97 |99 |344|320|24  |397|
|Kedia            |276|99 |79 |98 |277|318|-41 |376|
|Garde Nationale  |305|89 |91 |125|307|370|-63 |358|
|Police           |296|76 |85 |135|260|395|-135|313|
|Kaedi            |220|60 |63 |97 |218|330|-112|243|
|Douane           |108|49 |37 |22 |154|93 |61  |184|
|Armee            |202|40 |50 |112|172|338|-166|170|
|Chemal           |82 |35 |24 |23 |98 |73 |25  |129|
|Gendrim          |82 |27 |